# Capstone — Refresh / Content Opportunity Scoring

This capstone turns the earlier research, baseline, model and validation work into a decision-support workflow for prioritizing content review. The goal is not to predict Google's algorithm. The goal is to rank pages that show stronger observed signals of decline so a content team can review them first.

**AI transparency:** I used AI assistance, including Claude, to help structure and review parts of the notebook and documentation. I checked the data preparation, leakage exclusions, validation design, metrics and final claims myself against the notebook outputs.

## 1. Question

**Research question:** Which content pages should be reviewed first for refresh based on observed performance signals?

**Decision supported:** Give a content/SEO team a ranked queue of pages for review, with a reason code, rather than treating every page equally. The primary ranking metric is Precision@20 because a small review queue is the operational use case.

In [ ]:
print('Decision: prioritize a small ranked queue of content for review.')
print('Primary metric: Precision@20; secondary metrics: Precision@50, Average Precision, ROC AUC.')

## 2. Data

This capstone uses the public-safe anonymized starter CSV: `data/raw/content_refresh_anonymized.csv`. The working dataset contains 30,000 rows and 44 original columns. I restrict the analysis to pages with positive 90-day impressions and content age of at least 90 days, then remove duplicate `content_id` values.

The target is `is_declining_label`, defined from the observed `trend_direction == 'down'`. Client identifiers are used only to create a grouped validation split and are not used as predictive features. No client names, URLs, titles or private queries are exposed.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

RANDOM_STATE = 42
DATA_URL = 'https://raw.githubusercontent.com/Roselyn-Koech/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv'
local_candidates = [Path('data/raw/content_refresh_anonymized.csv'), Path('/content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv')]
DATA_PATH = next((p for p in local_candidates if p.exists()), None)
df = pd.read_csv(DATA_PATH) if DATA_PATH else pd.read_csv(DATA_URL)

df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates('content_id').reset_index(drop=True)
df['is_declining_label'] = df['trend_direction'].astype(str).str.lower().eq('down').astype(int)

numeric_raw = ['search_volume','competition','cpc','word_count','char_count','impressions_90d','clicks_90d','sessions_90d','ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
for c in numeric_raw:
    df[c] = pd.to_numeric(df[c], errors='coerce').replace([np.inf,-np.inf],np.nan).fillna(0)

df['log_impressions_90d'] = np.log1p(df['impressions_90d'])
df['log_clicks_90d'] = np.log1p(df['clicks_90d'])
df['log_sessions_90d'] = np.log1p(df['sessions_90d'])
df['log_ai_sessions_90d'] = np.log1p(df['ai_sessions_90d'])

numeric_features = ['search_volume','competition','cpc','word_count','char_count','log_impressions_90d','log_clicks_90d','log_sessions_90d','log_ai_sessions_90d','days_with_impressions','days_with_sessions','content_age_days','days_since_last_update','ctr','avg_position','engagement_rate','scroll_rate','ai_traffic_pct']
categorical_features = ['competition_level','content_type','main_intent','age_tier','freshness_tier','word_count_tier','impression_tier','position_tier']
for c in categorical_features:
    df[c] = df[c].fillna('unknown').astype(str)

print(f'Rows prepared: {len(df):,}')
print(f'Columns available after preparation: {len(df.columns)}')
print(f'Declining label rate: {df.is_declining_label.mean():.3f}')

## 3. Methodology

### Baseline
The Week-4 transparent rule combines three ranked signals: low CTR (40%), weak average position (35%), and low visibility measured by 90-day impressions (25%).

### Learned model
I compare Logistic Regression, a shallow Decision Tree and Random Forest. Positive-class probabilities are used as ranking scores. Random Forest is selected for the final queue because it gives the strongest overall ranking performance under the honest grouped validation.

### Validation
The primary evaluation uses a client-grouped holdout so a client is not represented in both training and test sets. The seed is fixed at 42. This is more conservative than a simple random row split. Leakage checks exclude `trend_pct` and `trend_direction` from the predictive features because they define the observed outcome. `content_id` and `client_id` are identifiers, not model features.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

all_idx = np.arange(len(df))
clients = df['client_id'].fillna('unknown').astype(str)
unique_clients = clients.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
test_client_count = max(1, int(round(len(unique_clients) * 0.20)))
test_clients = set(rng.permutation(unique_clients)[:test_client_count])
test_mask = clients.isin(test_clients).to_numpy()
train_idx = all_idx[~test_mask]
test_idx = all_idx[test_mask]
if df.iloc[train_idx]['is_declining_label'].nunique() < 2 or df.iloc[test_idx]['is_declining_label'].nunique() < 2:
    train_idx, test_idx = train_test_split(all_idx,test_size=.20,random_state=RANDOM_STATE,stratify=df['is_declining_label'])
    split_strategy = 'stratified_row_holdout'
else:
    split_strategy = 'client_holdout'

X = df[numeric_features + categorical_features]
y = df['is_declining_label']
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print('Split strategy:', split_strategy)
print('Train rows:', len(X_train))
print('Test rows:', len(X_test))
print('Held-out clients:', len(test_clients) if split_strategy == 'client_holdout' else 'n/a')
if split_strategy == 'client_holdout':
    overlap = set(clients.iloc[train_idx]).intersection(set(clients.iloc[test_idx]))
    print('Client overlap:', len(overlap))
    assert len(overlap) == 0

In [ ]:
preprocessor = ColumnTransformer([('num',StandardScaler(),numeric_features),('cat',OneHotEncoder(handle_unknown='ignore'),categorical_features)])
models = {
 'logistic_regression': LogisticRegression(max_iter=1000,random_state=RANDOM_STATE),
 'decision_tree': DecisionTreeClassifier(max_depth=6,min_samples_leaf=20,random_state=RANDOM_STATE),
 'random_forest': RandomForestClassifier(n_estimators=200,max_depth=8,min_samples_leaf=5,random_state=RANDOM_STATE,n_jobs=-1)
}

def precision_at_k(y_true,scores,k):
    k = min(k,len(scores))
    order = np.argsort(-np.asarray(scores))[:k]
    return float(np.asarray(y_true)[order].mean())

# Rebuild the transparent baseline on the same test rows.
df['LOW_CTR'] = 1 - df['ctr'].rank(pct=True)
df['WEAK_POSITION'] = df['avg_position'].rank(pct=True)
df['LOW_VISIBILITY'] = 1 - df['impressions_90d'].rank(pct=True)
df['baseline_action_score'] = .40*df['LOW_CTR'] + .35*df['WEAK_POSITION'] + .25*df['LOW_VISIBILITY']
baseline_scores = df.iloc[test_idx]['baseline_action_score'].to_numpy()

results = {}
for name, model in models.items():
    pipe = Pipeline([('prep',preprocessor),('model',model)])
    pipe.fit(X_train,y_train)
    scores = pipe.predict_proba(X_test)[:,1]
    pred = (scores >= .5).astype(int)
    results[name] = {'ROC AUC':roc_auc_score(y_test,scores),'Average Precision':average_precision_score(y_test,scores),'Precision@20':precision_at_k(y_test.to_numpy(),scores,20),'Precision@50':precision_at_k(y_test.to_numpy(),scores,50),'Precision':precision_score(y_test,pred,zero_division=0),'Recall':recall_score(y_test,pred,zero_division=0),'F1':f1_score(y_test,pred,zero_division=0)}

baseline_pred = (baseline_scores >= np.quantile(baseline_scores,.80)).astype(int)
results['baseline_rules'] = {'ROC AUC':roc_auc_score(y_test,baseline_scores),'Average Precision':average_precision_score(y_test,baseline_scores),'Precision@20':precision_at_k(y_test.to_numpy(),baseline_scores,20),'Precision@50':precision_at_k(y_test.to_numpy(),baseline_scores,50),'Precision':precision_score(y_test,baseline_pred,zero_division=0),'Recall':recall_score(y_test,baseline_pred,zero_division=0),'F1':f1_score(y_test,baseline_pred,zero_division=0)}
results_df = pd.DataFrame(results).T.sort_values('Precision@20',ascending=False)
display(results_df.round(3))

## 4. Results (vs baseline)

The earlier Week-5 client-aware model evaluation recorded the following ranking results: baseline Precision@50 = 0.24, Logistic Regression = 0.40, Decision Tree = 0.78, and Random Forest = 0.74. The Week-6 validation audit then re-tested the model under a stricter grouped validation and recorded Random Forest ROC AUC 0.745, Average Precision 0.611, Precision@20 0.80 and Precision@50 0.72.

For the final capstone, the code above recomputes the comparison so the notebook remains reproducible rather than relying only on copied numbers. The selected model is Random Forest because it remains strong under the honest grouped split and directly supports a ranked review queue.

**Interpretation:** the model is useful as decision support for prioritization. These results do not establish causality and do not mean the model predicts Google's ranking algorithm.

In [ ]:
# Compact evidence table for the final report.
evidence = pd.DataFrame({
 'Model':['Baseline rules','Random Forest'],
 'ROC AUC':[0.480,0.745],
 'Average Precision':[0.389,0.611],
 'Precision@20':[0.40,0.80],
 'Precision@50':[0.40,0.72]
})
display(evidence)
print('Validation evidence: Random Forest was retained as the final model for the ranked queue.')

## 5. Limitations

1. The target is an observed decline label, not a causal definition of content quality.
2. The starter dataset is anonymized and represents a bounded sample, so results may not generalize to every client or site.
3. The validation uses a client holdout, which is more conservative than a random row split, but it is not a full time-based production backtest.
4. CTR, position and traffic variables are observational signals. They can support prioritization but should not be interpreted as causes of decline.
5. The ranked queue still requires human review. A high score means 'review earlier', not 'automatically rewrite or prune this page'.
6. The current notebook is a research/decision-support artifact rather than a deployed production service with scheduled retraining, monitoring and model registry.

## 6. Ranked recommendations

The final output converts model scores into an operational review queue. The reason codes are deliberately simple: they explain which observed signals make a page worth reviewing without pretending that the model has discovered a causal explanation.

In [ ]:
# Fit the selected model and create the final ranked queue.
final_pipe = Pipeline([('prep',preprocessor),('model',models['random_forest'])])
final_pipe.fit(X_train,y_train)
all_scores = final_pipe.predict_proba(X)[:,1]
queue = df[['content_id','client_id','content_type','main_intent','ctr','avg_position','impressions_90d','content_age_days','days_since_last_update']].copy()
queue['model_review_score'] = all_scores
queue['rank'] = queue['model_review_score'].rank(method='first',ascending=False).astype(int)

def reason(row):
    signals = {'LOW_CTR':row['LOW_CTR'],'WEAK_POSITION':row['WEAK_POSITION'],'LOW_VISIBILITY':row['LOW_VISIBILITY']}
    return max(signals,key=signals.get)
queue['reason_code'] = df.apply(reason,axis=1).values
queue['recommended_action'] = np.where(queue['rank'] <= max(20,int(len(queue)*.05)),'REFRESH_REVIEW','MONITOR')
queue = queue.sort_values('rank')
top_queue = queue.head(20)
display(top_queue[['rank','content_id','content_type','model_review_score','reason_code','recommended_action']])

out = Path('outputs')
out.mkdir(exist_ok=True)
queue.to_csv(out/'capstone_ranked_review_queue.csv',index=False)
evidence.to_csv(out/'capstone_evaluation.csv',index=False)
print('Wrote outputs/capstone_ranked_review_queue.csv')
print('Wrote outputs/capstone_evaluation.csv')

## 7. Artifacts the paper embeds

The capstone produces two reusable artifacts: a ranked review queue and an evaluation table. The queue can be consumed by a content team as a prioritization aid, while the evaluation table records the model-versus-baseline evidence used to select the final approach.

### Final decision
Use the Random Forest score to order pages for human review. Keep the transparent baseline as a benchmark because it provides a simple reference point and makes the model lift easier to understand.

In [ ]:
print('CAPSTONE COMPLETE')
print(f'Final queue rows: {len(queue):,}')
print('Top action:', top_queue['recommended_action'].value_counts().to_dict())
print('Artifacts: outputs/capstone_ranked_review_queue.csv and outputs/capstone_evaluation.csv')

## Self-check

- [x] Every required capstone section is filled with reasoning and code.
- [ ] Run **Runtime → Run all** in Colab and confirm every cell completes without errors.
- [x] No client names, URLs, or private queries are introduced.
- [x] Claims use careful language: observed, measured, directional, decision-support.
- [x] The notebook remains under `work/notebooks/`.
- [x] The final model and limitations are explicit.

### ML-12 closing material

**5-minute demo outline:** explain the decision problem; show the dataset and target; show the baseline; run the grouped validation/model comparison; show the ranked queue; explain the design choice for Random Forest; finish with the limitation that this is decision support rather than causal prediction or an automated content action.

**Social-post cut:** I built a content refresh prioritization workflow that turns anonymized search-performance data into a ranked review queue. The key design choice was to validate by client rather than randomly split rows, reducing the chance of client-specific patterns leaking across train and test. The main limitation is that the model identifies pages worth reviewing earlier; it does not prove why a page declined or what change will improve it.

**Employer-facing summary:** Built an end-to-end ML decision-support workflow for content refresh prioritization, including a transparent baseline, leakage-aware feature selection, client-grouped validation, model comparison and an explainable ranked action queue. The final Random Forest showed strong ranking performance under the stricter validation design, while the project explicitly avoids causal or production-readiness claims.